In [ ]:
from collections import deque
from typing import List

def findOptimalCommute(grid: List[List[str]], modes: List[str],
                       times: List[int], costs: List[int]) -> str:
    rows, cols = len(grid), len(grid[0])

    # Find start and destination
    start, dest = None, None
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'S':
                start = (r, c)
            elif grid[r][c] == 'D':
                dest = (r, c)

    if not start or not dest:
        return ""

    best_time = float('inf')
    best_cost = float('inf')
    best_mode = ""

    # Try each transportation mode one by one
    for mode_idx in range(len(modes)):
        mode_digit = str(mode_idx + 1)

        # BFS for this specific mode
        queue = deque([(start[0], start[1], 0)])  # (row, col, distance)
        visited = {start}
        found = False

        while queue and not found:
            r, c, dist = queue.popleft()

            # Check if we reached destination
            if (r, c) == dest:
                # Calculate time and cost
                total_time = dist * times[mode_idx]
                total_cost = dist * costs[mode_idx]

                # Update best mode if this is faster or cheaper
                if (total_time < best_time or
                    (total_time == best_time and total_cost < best_cost)):
                    best_time = total_time
                    best_cost = total_cost
                    best_mode = modes[mode_idx]
                found = True
                break

            # Check neighbors
            for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                nr, nc = r + dr, c + dc

                if (0 <= nr < rows and 0 <= nc < cols and
                    (nr, nc) not in visited):
                    cell = grid[nr][nc]

                    # Move if cell matches our mode or is destination
                    if cell == mode_digit or cell == 'D':
                        visited.add((nr, nc))
                        # Don't add distance for D (it's free)
                        new_dist = dist if cell == 'D' else dist + 1
                        queue.append((nr, nc, new_dist))

    return best_mode


In [ ]:
### Optimal Compute

from collections import deque
from typing import List

def findOptimalCommuteOptimized(grid: List[List[str]], modes: List[str],
                                times: List[int], costs: List[int]) -> str:
    rows, cols = len(grid), len(grid[0])

    # Find start and destination
    start, dest = None, None
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'S':
                start = (r, c)
            elif grid[r][c] == 'D':
                dest = (r, c)

    if not start or not dest:
        return ""

    best_time = float('inf')
    best_cost = float('inf')
    best_mode = ""

    # Single BFS: (row, col, mode_used, distance)
    # mode_used is the number string of the mode
    queue = deque()
    visited = set()

    # Initialize: check all neighbors of start
    for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        nr, nc = start[0] + dr, start[1] + dc

        if 0 <= nr < rows and 0 <= nc < cols:
            cell = grid[nr][nc]

            if cell.isdigit():
                mode_digit = cell
                visited.add((nr, nc, mode_digit))
                queue.append((nr, nc, mode_digit, 1))

    # Run BFS
    while queue:
        r, c, mode_digit, dist = queue.popleft()

        # Check if we reached destination
        if grid[r][c] == 'D':
            mode_idx = int(mode_digit) - 1
            total_time = dist * times[mode_idx]
            total_cost = dist * costs[mode_idx]

            # Update best mode
            if (total_time < best_time or
                (total_time == best_time and total_cost < best_cost)):
                best_time = total_time
                best_cost = total_cost
                best_mode = modes[mode_idx]
            continue

        # Look at neighbors using the SAME mode
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nr, nc = r + dr, c + dc

            if (0 <= nr < rows and 0 <= nc < cols and
                (nr, nc, mode_digit) not in visited):
                cell = grid[nr][nc]

                # Move if same mode or destination
                if cell == mode_digit or cell == 'D':
                    visited.add((nr, nc, mode_digit))
                    # Don't add distance for D
                    new_dist = dist if cell == 'D' else dist + 1
                    queue.append((nr, nc, mode_digit, new_dist))

    return best_mode




In [ ]:
import heapq
from typing import List

def findOptimalCommuteWithSwitching(grid: List[List[str]], modes: List[str],
                                    times: List[int], costs: List[int],
                                    switch_time: int, switch_cost: int) -> str:
    rows, cols = len(grid), len(grid[0])

    # Find start and destination
    start, dest = None, None
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'S':
                start = (r, c)
            elif grid[r][c] == 'D':
                dest = (r, c)

    if not start or not dest:
        return ""

    # Priority queue: (total_time, total_cost, row, col, mode_idx)
    pq = []

    # Initialize with all modes from start
    for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        nr, nc = start[0] + dr, start[1] + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            cell = grid[nr][nc]
            if cell.isdigit():
                mode_idx = int(cell) - 1
                heapq.heappush(pq, (times[mode_idx], costs[mode_idx], nr, nc, mode_idx))

    # Track best (time, cost) for each (row, col, mode)
    best = {}

    while pq:
        curr_time, curr_cost, r, c, mode_idx = heapq.heappop(pq)

        # Check if we've seen this state with better time/cost
        state = (r, c, mode_idx)
        if state in best:
            best_time, best_cost = best[state]
            if (curr_time > best_time or
                (curr_time == best_time and curr_cost >= best_cost)):
                continue

        best[state] = (curr_time, curr_cost)

        # Check if reached destination
        if grid[r][c] == 'D':
            return modes[mode_idx]

        # Explore neighbors
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nr, nc = r + dr, c + dc

            if 0 <= nr < rows and 0 <= nc < cols:
                cell = grid[nr][nc]

                if cell == 'X':
                    continue

                # Special case: destination
                if cell == 'D':
                    # Move to destination with current mode (no extra cost)
                    next_state = (nr, nc, mode_idx)
                    if next_state in best:
                        best_time, best_cost = best[next_state]
                        if (curr_time > best_time or
                            (curr_time == best_time and curr_cost >= best_cost)):
                            continue
                    heapq.heappush(pq, (curr_time, curr_cost, nr, nc, mode_idx))
                    continue

                # Try all possible modes for next cell
                for next_mode_idx in range(len(modes)):
                    next_mode_digit = str(next_mode_idx + 1)

                    # Check if this mode is valid for the cell
                    if cell != next_mode_digit:
                        continue

                    # Calculate new time and cost
                    new_time = curr_time + times[next_mode_idx]
                    new_cost = curr_cost + costs[next_mode_idx]

                    # Add switch penalty if changing modes
                    if next_mode_idx != mode_idx:
                        new_time += switch_time
                        new_cost += switch_cost

                    # Check if this is better than previous visit
                    next_state = (nr, nc, next_mode_idx)
                    if next_state in best:
                        best_time, best_cost = best[next_state]
                        if (new_time > best_time or
                            (new_time == best_time and new_cost >= best_cost)):
                            continue

                    heapq.heappush(pq, (new_time, new_cost, nr, nc, next_mode_idx))

    return ""  # No path found